In [1]:
import os
import psycopg
import pandas as pd
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, 
    SplineTransformer, 
    QuantileTransformer, 
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)

In [2]:
TABLE_NAME = "clean_users_churn"

In [3]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

In [4]:
EXPERIMENT_NAME = "WORK_WITH_FEATURES"
RUN_NAME = "preprocessing"
REGISTRY_MODEL_NAME = "PREPROCESSED_MODEL"

In [5]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credits = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD")
}

connection.update(postgres_credits)

In [6]:
with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        data = cur.fetchall()

        columns = [column[0] for column in cur.description]

df = pd.DataFrame(data=data, columns=columns)

In [7]:
df

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,28,1680-VDCWW,2019-02-01,NaT,One year,No,Bank transfer (automatic),19.80,202.25,Fiber optic,...,No,No,No,No,Male,0,Yes,No,No,0
1,29,1066-JKSGK,2019-11-01,2019-12-01,Month-to-month,No,Mailed check,20.15,20.15,Fiber optic,...,No,No,No,No,Male,0,No,No,No,1
2,30,3638-WEABW,2015-04-01,NaT,Two year,Yes,Credit card (automatic),59.90,3505.10,DSL,...,No,Yes,No,No,Female,0,Yes,No,Yes,0
3,31,6322-HRPFA,2016-01-01,NaT,Month-to-month,No,Credit card (automatic),59.60,2970.30,DSL,...,No,Yes,No,No,Male,0,Yes,Yes,No,0
4,32,6865-JZNKO,2017-08-01,NaT,Month-to-month,Yes,Bank transfer (automatic),55.30,1530.60,DSL,...,No,No,No,No,Female,0,No,No,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7019,6989,1264-FUHCX,2018-03-01,NaT,Month-to-month,Yes,Credit card (automatic),69.50,1652.10,DSL,...,No,Yes,Yes,No,Female,0,Yes,Yes,Yes,0
7020,6990,7789-CRUVC,2018-06-01,NaT,One year,No,Bank transfer (automatic),76.00,1588.75,DSL,...,No,No,Yes,Yes,Female,0,Yes,Yes,Yes,0
7021,6991,6598-KELSS,2017-02-01,NaT,Month-to-month,Yes,Electronic check,93.60,3366.05,Fiber optic,...,No,No,Yes,Yes,Male,0,Yes,No,No,0
7022,6992,8739-WWKDU,2019-03-01,2019-11-01,Month-to-month,No,Bank transfer (automatic),95.65,778.10,Fiber optic,...,No,No,Yes,Yes,Female,0,No,No,Yes,1


In [8]:
obj_df = df.select_dtypes(include=["object"])

In [7]:
cat_columns = ["type", "payment_method", "internet_service", "gender"]

In [ ]:
encoder_oh = OneHotEncoder(categories="auto", handle_unknown="error", max_categories=15, drop="first", sparse_output=False)

In [9]:
num_columns = ["monthly_charges", "total_charges"]
n_knots = 3
degree_spline = 4
n_quantiles=100
degree = 3
n_bins = 5
encode = 'ordinal'
strategy = 'uniform'
subsample = None

In [10]:
# SplineTransformer
encoder_spl = SplineTransformer(n_knots=n_knots, degree=degree_spline)

# QuantileTransformer
encoder_q = QuantileTransformer(n_quantiles=n_quantiles)

# RobustScaler
encoder_rb = RobustScaler()

# PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)

# KBinsDiscretizer
encoder_kbd = KBinsDiscretizer(n_bins=n_bins, encode=encode, strategy=strategy, subsample=subsample)


In [11]:
numeric_transformer = ColumnTransformer(
    transformers=[
        ("spl", encoder_spl, num_columns),
        ("q", encoder_q, num_columns),
        ("rb", encoder_rb, num_columns),
        ("pol", encoder_pol, num_columns),
        ("kbd", encoder_kbd, num_columns),
]
)

categorical_transformer = Pipeline(steps=[('encoder', encoder_oh)])

preprocessor = ColumnTransformer(
	    transformers=[
            ("num", numeric_transformer, num_columns),
            ("cat", categorical_transformer, cat_columns)
        ],
    n_jobs=-1
)

encoded_features = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(encoded_features, columns=preprocessor.get_feature_names_out())

df = pd.concat([df, transformed_df], axis=1)

In [1]:
df

NameError: name 'df' is not defined

In [23]:
preprocessor

ColumnTransformer(n_jobs=-1,
                  transformers=[('num',
                                 ColumnTransformer(transformers=[('spl',
                                                                  SplineTransformer(degree=4,
                                                                                    n_knots=3),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('q',
                                                                  QuantileTransformer(n_quantiles=100),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('rb',
                                                                  RobustScaler(),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('pol',
                                                                  PolynomialFeatures(degree=3),
                                                                  ['monthly_charges',
                                                                   'total_charges']),
                                                                 ('kbd',
                                                                  KBinsDiscretizer(encode='ordinal',
                                                                                   strategy='uniform',
                                                                                   subsample=None),
                                                                  ['monthly_charges',
                                                                   'total_charges'])]),
                                 ['monthly_charges', 'total_charges']),
                                ('cat',
                                 Pipeline(steps=[('encoder',
                                                  OneHotEncoder(drop='first',
                                                                max_categories=15,
                                                                sparse_output=False))]),
                                 ['type', 'payment_method', 'internet_service',
                                  'gender'])])

In [ ]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"


In [13]:
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [18]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

In [17]:
if not experiment_id:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

In [19]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.sklearn.log_model(preprocessor, "column_transformer")

2026/01/14 07:18:07 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [21]:
df

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,num__pol__total_charges^3,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__gender_Male
0,28,1680-VDCWW,2019-02-01,NaT,One year,No,Bank transfer (automatic),19.80,202.25,Fiber optic,...,8.273049e+06,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
1,29,1066-JKSGK,2019-11-01,2019-12-01,Month-to-month,No,Mailed check,20.15,20.15,Fiber optic,...,8.181353e+03,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,30,3638-WEABW,2015-04-01,NaT,Two year,Yes,Credit card (automatic),59.90,3505.10,DSL,...,4.306270e+10,2.0,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3,31,6322-HRPFA,2016-01-01,NaT,Month-to-month,No,Credit card (automatic),59.60,2970.30,DSL,...,2.620601e+10,2.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,32,6865-JZNKO,2017-08-01,NaT,Month-to-month,Yes,Bank transfer (automatic),55.30,1530.60,DSL,...,3.585792e+09,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7019,6989,1264-FUHCX,2018-03-01,NaT,Month-to-month,Yes,Credit card (automatic),69.50,1652.10,DSL,...,4.509299e+09,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7020,6990,7789-CRUVC,2018-06-01,NaT,One year,No,Bank transfer (automatic),76.00,1588.75,DSL,...,4.010206e+09,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7021,6991,6598-KELSS,2017-02-01,NaT,Month-to-month,Yes,Electronic check,93.60,3366.05,Fiber optic,...,3.813833e+10,3.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0
7022,6992,8739-WWKDU,2019-03-01,2019-11-01,Month-to-month,No,Bank transfer (automatic),95.65,778.10,Fiber optic,...,4.710926e+08,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [27]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [28]:
X_train, X_test, y_train, y_test = train_test_split(df, df["target"])

In [30]:
y_train

5404    0
6954    1
2510    0
6618    0
5841    0
       ..
5050    0
2241    0
5458    0
4659    0
2915    1
Name: target, Length: 5268, dtype: int64

In [29]:
X_train

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,num__pol__total_charges^3,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__gender_Male
5404,5599,5546-BYZSM,2017-07-01,NaT,One year,Yes,Credit card (automatic),75.50,2424.45,DSL,...,1.425081e+10,2.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
6954,4991,4482-FTFFX,2016-12-01,2019-12-01,Month-to-month,Yes,Bank transfer (automatic),97.35,3457.90,Fiber optic,...,4.134636e+10,3.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2510,2614,1265-BCFEO,2014-02-01,NaT,Two year,Yes,Electronic check,114.90,8496.70,Fiber optic,...,6.134100e+11,4.0,4.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
6618,6853,1036-GUDCL,2014-08-01,NaT,Two year,No,Bank transfer (automatic),65.70,4378.90,DSL,...,8.396438e+10,2.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5841,6089,3055-OYMSE,2020-01-01,NaT,Month-to-month,Yes,Credit card (automatic),49.80,49.80,DSL,...,1.235060e+05,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5050,5246,2282-YGNOR,2017-12-01,NaT,Month-to-month,Yes,Credit card (automatic),54.75,1406.90,DSL,...,2.784772e+09,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2241,2342,2774-LVQUS,2018-01-01,NaT,One year,Yes,Mailed check,26.80,733.55,Fiber optic,...,3.947200e+08,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
5458,5654,9801-GDWGV,2017-10-01,NaT,One year,Yes,Electronic check,39.10,1096.60,DSL,...,1.318696e+09,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4659,4810,0366-NQSHS,2015-08-01,NaT,Two year,No,Electronic check,80.60,4299.95,DSL,...,7.950423e+10,3.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [24]:
model = CatBoostClassifier(iterations=1000,
                           learning_rate=0.01,
                           )

In [25]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

In [31]:
pipeline.fit(X_train, y_train)

0:	learn: 0.6862796	total: 60.4ms	remaining: 1m
1:	learn: 0.6797384	total: 67.4ms	remaining: 33.6s
2:	learn: 0.6730482	total: 74.4ms	remaining: 24.7s
3:	learn: 0.6668782	total: 81.3ms	remaining: 20.2s
4:	learn: 0.6608446	total: 88.2ms	remaining: 17.5s
5:	learn: 0.6555562	total: 95.8ms	remaining: 15.9s
6:	learn: 0.6508641	total: 103ms	remaining: 14.5s
7:	learn: 0.6449942	total: 109ms	remaining: 13.5s
8:	learn: 0.6395732	total: 116ms	remaining: 12.8s
9:	learn: 0.6342160	total: 123ms	remaining: 12.1s
10:	learn: 0.6290411	total: 130ms	remaining: 11.7s
11:	learn: 0.6240464	total: 137ms	remaining: 11.2s
12:	learn: 0.6196379	total: 143ms	remaining: 10.9s
13:	learn: 0.6156811	total: 149ms	remaining: 10.5s
14:	learn: 0.6117194	total: 156ms	remaining: 10.2s
15:	learn: 0.6071651	total: 162ms	remaining: 9.99s
16:	learn: 0.6026404	total: 169ms	remaining: 9.78s
17:	learn: 0.5977879	total: 178ms	remaining: 9.7s
18:	learn: 0.5931686	total: 184ms	remaining: 9.5s
19:	learn: 0.5897005	total: 190ms	remain

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  ColumnTransformer(transformers=[('spl',
                                                                                   SplineTransformer(degree=4,
                                                                                                     n_knots=3),
                                                                                   ['monthly_charges',
                                                                                    'total_charges']),
                                                                                  ('q',
                                                                                   QuantileTransformer(n_quantiles=100),
                                                                                   ['monthly_charges',
                                                                                    'total_charges']),
                                                                                  ('rb',
                                                                                   RobustScaler(),
                                                                                   ['monthly_charges',
                                                                                    'total_charges']),
                                                                                  ('pol',
                                                                                   Polynomial...
                                                                                   KBinsDiscretizer(encode='ordinal',
                                                                                                    strategy='uniform',
                                                                                                    subsample=None),
                                                                                   ['monthly_charges',
                                                                                    'total_charges'])]),
                                                  ['monthly_charges',
                                                   'total_charges']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 max_categories=15,
                                                                                 sparse_output=False))]),
                                                  ['type', 'payment_method',
                                                   'internet_service',
                                                   'gender'])])),
                ('model',
                 <catboost.core.CatBoostClassifier object at 0x7f82508eaa70>)])

In [33]:
prediction = pipeline.predict(X_test)

In [37]:
import yaml, joblib, json
from sklearn.model_selection import KFold, cross_validate

In [40]:
cv_strategy = KFold(n_splits=5, shuffle=True)
cv_res = cross_validate(pipeline, X_test, y_test, cv=cv_strategy, n_jobs=-1, scoring=['neg_mean_squared_log_error', 'r2'])

0:	learn: 0.6875014	total: 53ms	remaining: 52.9s
1:	learn: 0.6819499	total: 58.5ms	remaining: 29.2s
2:	learn: 0.6754661	total: 63.7ms	remaining: 21.2s
0:	learn: 0.6881158	total: 56.2ms	remaining: 56.1s
3:	learn: 0.6695242	total: 69ms	remaining: 17.2s
1:	learn: 0.6826670	total: 61.6ms	remaining: 30.7s
4:	learn: 0.6640559	total: 78.9ms	remaining: 15.7s
2:	learn: 0.6765882	total: 72.9ms	remaining: 24.2s
5:	learn: 0.6590171	total: 90.6ms	remaining: 15s
3:	learn: 0.6713817	total: 83.2ms	remaining: 20.7s
6:	learn: 0.6545178	total: 96.1ms	remaining: 13.6s
4:	learn: 0.6660787	total: 97.6ms	remaining: 19.4s
7:	learn: 0.6499590	total: 109ms	remaining: 13.5s
8:	learn: 0.6445171	total: 114ms	remaining: 12.6s
5:	learn: 0.6605997	total: 113ms	remaining: 18.7s
6:	learn: 0.6552721	total: 119ms	remaining: 16.8s
7:	learn: 0.6508821	total: 124ms	remaining: 15.4s
9:	learn: 0.6398644	total: 126ms	remaining: 12.5s
10:	learn: 0.6351415	total: 132ms	remaining: 11.8s
8:	learn: 0.6464984	total: 137ms	remaining:

In [42]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    model_info = mlflow.sklearn.log_model(sk_model=pipeline,
                            artifact_path="models",
                            registered_model_name=REGISTRY_MODEL_NAME,
                            )

Registered model 'PREPROCESSED_MODEL' already exists. Creating a new version of this model...
2026/01/14 09:41:12 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: PREPROCESSED_MODEL, version 2
Created version '2' of model 'PREPROCESSED_MODEL'.


In [45]:
from mlflow import MlflowClient


In [46]:
client = MlflowClient()

In [49]:
EXP = client.get_experiment_by_name(EXPERIMENT_NAME)

In [54]:
versions = client.search_model_versions(f"name='{REGISTRY_MODEL_NAME}'")

In [58]:
run_id

'2b32f528461c4516854050eb4fabfd7b'